# 🏦 Clustering Clients Fintech — Pipeline Complet & Pédagogique

**Objectif** : Segmenter les clients d'une application fintech à partir de leur comportement transactionnel.  
Ce notebook est conçu pour être **pédagogique**, **reproductible** (Colab + local) et **facilement extensible**.

---

## 📋 Plan du notebook

| Étape | Description |
|-------|-------------|
| 0 | Installation & Configuration de l'environnement |
| 1 | Ingestion des données |
| 2 | Exploration & Nettoyage (EDA) |
| 3 | Feature Engineering (RFM + comportement transactionnel) |
| 4 | Prétraitement (scaling, réduction bruit) |
| 5 | Comparatif de modèles de clustering |
| 6 | Évaluation des clusters |
| 7 | Visualisations (PCA / UMAP / profils) |
| 8 | Interprétation métier & synthèse |

---

> 💡 **Comment utiliser ce notebook ?**
> - Exécutez les cellules dans l'ordre (Run All ou Ctrl+Enter cellule par cellule).
> - Les sections marquées `⚙️ OPTIONNEL` peuvent être sautées sans impacter le reste.
> - Pour utiliser **vos propres données**, remplacez la section *1 – Ingestion* par votre fichier CSV.

---
## ⚙️ Étape 0 — Installation & Configuration

In [ ]:
# ── Détection de l'environnement (Colab vs local) ────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f"Environnement détecté : {'Google Colab' if IN_COLAB else 'Local'}")

# ── Installation des dépendances (uniquement si manquantes) ──────────────────
import importlib

REQUIRED = {
    'hdbscan': 'hdbscan',
    'umap': 'umap-learn',
    'plotly': 'plotly',
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installation des packages manquants : {missing}")
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
    print("✅ Installation terminée.")
else:
    print("✅ Tous les packages sont déjà installés.")

In [ ]:
# ── Imports principaux ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from datetime import datetime, timedelta

# Sklearn — preprocessing
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Sklearn — clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# Sklearn — métriques
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# HDBSCAN
import hdbscan

# UMAP (optionnel)
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️  umap-learn non disponible — les visualisations UMAP seront ignorées.")

# ── Configuration globale ─────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Palette de couleurs pour les clusters
CLUSTER_PALETTE = sns.color_palette('tab10', 10)

print("✅ Imports réussis.")

---
## 📥 Étape 1 — Ingestion des données

Deux modes disponibles :
- **Mode A** : chargement depuis votre propre fichier CSV (remplacez `DATA_PATH`).
- **Mode B** : génération d'un dataset synthétique réaliste (activé par défaut).

> ⚠️ Les données synthétiques imitent un jeu de transactions fintech typique mais ne représentent pas de vrais clients.

In [ ]:
# ── Configuration du chemin de données ───────────────────────────────────────
# Pour utiliser votre propre fichier :
#   1. Mettez USE_SYNTHETIC_DATA = False
#   2. Indiquez le chemin vers votre CSV dans DATA_PATH
#      (sur Colab : uploadez d'abord via l'onglet Fichiers ou drive.mount)

USE_SYNTHETIC_DATA = True   # ← mettre False pour utiliser vos données

# Chemin du fichier CSV (ignoré si USE_SYNTHETIC_DATA=True)
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'transactions.csv')

# Colonnes attendues dans votre CSV (adaptez selon votre jeu de données)
EXPECTED_COLUMNS = [
    'customer_id',   # identifiant unique du client
    'transaction_date',  # date de la transaction (format YYYY-MM-DD)
    'amount',        # montant de la transaction
    'transaction_type',  # 'credit' ou 'debit'
    'category',      # catégorie marchande (optionnel)
    'country',       # pays (optionnel)
]

In [ ]:
def generate_synthetic_fintech_data(n_customers: int = 2000,
                                    n_transactions: int = 50000,
                                    seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Génère un dataset transactionnel synthétique avec 4 profils de clients :
      - Profil 0 : clients inactifs / petits volumes
      - Profil 1 : clients actifs standard
      - Profil 2 : clients premium (gros volumes, fréquents)
      - Profil 3 : clients à risque (transactions irrégulières, montants extrêmes)
    """
    rng = np.random.default_rng(seed)

    # Profils cachés (ground truth, pour validation)
    profile_weights = [0.35, 0.40, 0.15, 0.10]
    profiles = rng.choice(4, size=n_customers, p=profile_weights)

    # Paramètres par profil
    profile_params = {
        0: dict(tx_rate=0.3, avg_amount=50,   std_amount=30,  international_rate=0.05),
        1: dict(tx_rate=1.0, avg_amount=150,  std_amount=80,  international_rate=0.15),
        2: dict(tx_rate=3.0, avg_amount=600,  std_amount=300, international_rate=0.30),
        3: dict(tx_rate=0.8, avg_amount=200,  std_amount=500, international_rate=0.40),
    }

    categories  = ['food', 'transport', 'entertainment', 'health', 'shopping', 'travel']
    countries   = ['FR', 'US', 'DE', 'GB', 'MA', 'SN', 'CI', 'CM']
    home_country = 'FR'

    records = []
    reference_date = datetime(2024, 12, 1)

    for cid in range(n_customers):
        p = profiles[cid]
        params = profile_params[p]
        n_tx = max(1, int(rng.poisson(params['tx_rate'] * 25)))

        for _ in range(n_tx):
            days_back = int(rng.integers(1, 365))
            tx_date   = reference_date - timedelta(days=days_back)

            amount = abs(rng.normal(params['avg_amount'], params['std_amount']))
            amount = round(max(0.01, amount), 2)

            is_international = rng.random() < params['international_rate']
            country = rng.choice([c for c in countries if c != home_country]) \
                      if is_international else home_country

            tx_type   = rng.choice(['credit', 'debit'], p=[0.3, 0.7])
            category  = rng.choice(categories)

            records.append({
                'customer_id':      f'C{cid:05d}',
                'transaction_date': tx_date.strftime('%Y-%m-%d'),
                'amount':           amount,
                'transaction_type': tx_type,
                'category':         category,
                'country':          country,
                'true_profile':     p,  # pour évaluation, retiré en production
            })

    df = pd.DataFrame(records)
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    return df.sort_values('transaction_date').reset_index(drop=True)


# ── Chargement / Génération ───────────────────────────────────────────────────
if USE_SYNTHETIC_DATA:
    print("📦 Génération du dataset synthétique...")
    df_transactions = generate_synthetic_fintech_data()
    print(f"✅ Dataset généré : {df_transactions.shape[0]:,} transactions — {df_transactions['customer_id'].nunique():,} clients")
else:
    print(f"📂 Chargement depuis : {DATA_PATH}")
    df_transactions = pd.read_csv(DATA_PATH, parse_dates=['transaction_date'])
    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df_transactions.columns]
    if missing_cols:
        raise ValueError(f"Colonnes manquantes dans votre CSV : {missing_cols}")
    print(f"✅ Données chargées : {df_transactions.shape}")

df_transactions.head()

---
## 🔍 Étape 2 — Exploration & Nettoyage (EDA)

Avant de construire les features, nous devons :
1. Comprendre la **structure** du jeu de données.
2. Détecter les **valeurs manquantes** et les **outliers extrêmes**.
3. Effectuer un nettoyage minimal.

In [ ]:
print("=" * 60)
print("APERÇU DU DATASET")
print("=" * 60)
print(f"Dimensions          : {df_transactions.shape}")
print(f"Période             : {df_transactions['transaction_date'].min().date()} → {df_transactions['transaction_date'].max().date()}")
print(f"Clients uniques     : {df_transactions['customer_id'].nunique():,}")
print()
print(df_transactions.dtypes)
print()
print("Valeurs manquantes :")
print(df_transactions.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution des montants (log)
axes[0].hist(np.log1p(df_transactions['amount']), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution log(montant + 1)')
axes[0].set_xlabel('log(amount + 1)')
axes[0].set_ylabel('Fréquence')

# Répartition type de transaction
tx_counts = df_transactions['transaction_type'].value_counts()
axes[1].bar(tx_counts.index, tx_counts.values, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Type de transactions')
axes[1].set_ylabel('Nombre')

# Nombre de transactions par client
n_tx_per_client = df_transactions.groupby('customer_id').size()
axes[2].hist(n_tx_per_client, bins=40, color='darkorange', edgecolor='white')
axes[2].set_title('Transactions par client')
axes[2].set_xlabel('Nb transactions')
axes[2].set_ylabel('Nb clients')

plt.tight_layout()
plt.suptitle('Exploration des données brutes', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ── Nettoyage ─────────────────────────────────────────────────────────────────
print("Nettoyage des données...")

df_clean = df_transactions.copy()

# 1. Supprimer les lignes avec montant nul ou négatif
mask_invalid = df_clean['amount'] <= 0
print(f"  → Suppression de {mask_invalid.sum()} transactions avec montant ≤ 0")
df_clean = df_clean[~mask_invalid]

# 2. Winsorisation des montants (cap au 99e percentile)
cap_99 = df_clean['amount'].quantile(0.99)
n_capped = (df_clean['amount'] > cap_99).sum()
df_clean['amount'] = df_clean['amount'].clip(upper=cap_99)
print(f"  → Winsorisation : {n_capped} montants plafonnés à {cap_99:.2f}")

# 3. Uniformisation des types textuels
df_clean['transaction_type'] = df_clean['transaction_type'].str.lower().str.strip()
df_clean['category']         = df_clean['category'].str.lower().str.strip()

# 4. Supprimer les doublons exacts
n_dup = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"  → Suppression de {n_dup} doublons")

print(f"\n✅ Après nettoyage : {df_clean.shape[0]:,} transactions — {df_clean['customer_id'].nunique():,} clients")

---
## 🔧 Étape 3 — Feature Engineering

Les modèles de clustering ne peuvent pas travailler directement sur des transactions.  
Nous devons **agréger** les données au niveau **client** pour créer des features descriptives.

### Features construites

#### 📊 RFM (Recency – Frequency – Monetary)
| Feature | Description |
|---------|-------------|
| `recency_days` | Nombre de jours depuis la dernière transaction |
| `frequency`    | Nombre total de transactions |
| `monetary`     | Montant total dépensé |

#### 💳 Comportement transactionnel
| Feature | Description |
|---------|-------------|
| `avg_amount`           | Montant moyen par transaction |
| `std_amount`           | Variabilité des montants |
| `max_amount`           | Montant maximum observé |
| `debit_ratio`          | Part des transactions de débit |
| `international_ratio`  | Part des transactions internationales |
| `active_days`          | Nombre de jours d'activité distincts |
| `avg_tx_per_day`       | Fréquence quotidienne moyenne |

#### 🚨 Indicateurs d'anomalie
| Feature | Description |
|---------|-------------|
| `amount_cv`        | Coefficient de variation des montants (std/mean) |
| `top_category`     | Catégorie la plus fréquente (encodée) |
| `n_unique_countries` | Nombre de pays distincts |


In [ ]:
def build_customer_features(df: pd.DataFrame,
                            reference_date: datetime = None) -> pd.DataFrame:
    """
    Agrège les transactions au niveau client et calcule
    les features RFM + comportementales + anomalie.
    """
    if reference_date is None:
        reference_date = df['transaction_date'].max() + timedelta(days=1)

    grp = df.groupby('customer_id')

    # ── RFM ──────────────────────────────────────────────────────────────────
    rfm = pd.DataFrame({
        'recency_days': (reference_date - grp['transaction_date'].max()).dt.days,
        'frequency':    grp['amount'].count(),
        'monetary':     grp['amount'].sum().round(2),
    })

    # ── Comportement transactionnel ───────────────────────────────────────────
    behaviour = pd.DataFrame({
        'avg_amount':    grp['amount'].mean().round(2),
        'std_amount':    grp['amount'].std().fillna(0).round(2),
        'max_amount':    grp['amount'].max().round(2),
        'median_amount': grp['amount'].median().round(2),
        'debit_ratio':   grp.apply(
            lambda x: (x['transaction_type'] == 'debit').mean()
        ).round(4),
        'international_ratio': grp.apply(
            lambda x: (x['country'] != 'FR').mean()
        ).round(4),
        'active_days':  grp['transaction_date'].nunique(),
        'n_unique_countries': grp['country'].nunique(),
    })

    # Fréquence quotidienne moyenne
    date_range = (grp['transaction_date'].max() - grp['transaction_date'].min()).dt.days + 1
    behaviour['avg_tx_per_day'] = (rfm['frequency'] / date_range).round(4)

    # ── Indicateurs d'anomalie ────────────────────────────────────────────────
    anomaly = pd.DataFrame({
        'amount_cv': (behaviour['std_amount'] / behaviour['avg_amount'].replace(0, np.nan)).fillna(0).round(4),
    })

    # Catégorie dominante (encodée ordinalement par fréquence globale)
    cat_order = df['category'].value_counts().index.tolist()
    top_cat   = grp['category'].agg(lambda x: x.mode()[0] if len(x) > 0 else 'unknown')
    anomaly['top_category_encoded'] = top_cat.map({c: i for i, c in enumerate(cat_order)}).fillna(-1).astype(int)

    # ── Fusion ───────────────────────────────────────────────────────────────
    features = rfm.join(behaviour).join(anomaly)

    # Profil de vérité terrain (optionnel — disponible seulement en mode synthétique)
    if 'true_profile' in df.columns:
        features['true_profile'] = grp['true_profile'].first()

    return features.reset_index()


print("⚙️  Construction des features clients...")
df_features = build_customer_features(df_clean)
print(f"✅ Features construites : {df_features.shape[0]} clients × {df_features.shape[1]} colonnes")
print()
display(df_features.describe().round(2))

In [ ]:
# ── Matrice de corrélation ────────────────────────────────────────────────────
numeric_cols = df_features.select_dtypes(include='number').columns.tolist()
exclude_cols = ['true_profile', 'top_category_encoded']
plot_cols    = [c for c in numeric_cols if c not in exclude_cols]

corr_matrix = df_features[plot_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', linewidths=0.5,
)
plt.title('Matrice de corrélation des features clients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚖️ Étape 4 — Prétraitement des features

Le clustering est très sensible à l'**échelle** des variables.  
Nous appliquons un **RobustScaler** (résistant aux outliers) plutôt qu'un StandardScaler classique.

> **Pourquoi RobustScaler ?**  
> En fintech, les distributions de montants sont fortement asymétriques (*skewed*).  
> RobustScaler utilise la médiane et l'IQR plutôt que la moyenne et l'écart-type, le rendant moins sensible aux valeurs extrêmes.

In [ ]:
# ── Sélection des features pour le clustering ─────────────────────────────────
FEATURE_COLS = [
    'recency_days',
    'frequency',
    'monetary',
    'avg_amount',
    'std_amount',
    'max_amount',
    'debit_ratio',
    'international_ratio',
    'active_days',
    'avg_tx_per_day',
    'amount_cv',
    'n_unique_countries',
]

X_raw = df_features[FEATURE_COLS].copy()

# Vérification des NaN résiduels
n_nan = X_raw.isnull().sum().sum()
if n_nan > 0:
    print(f"⚠️  {n_nan} valeurs NaN détectées — remplacement par la médiane")
    for col in X_raw.columns:
        X_raw[col] = X_raw[col].fillna(X_raw[col].median())

# Scaling
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

print(f"✅ Données normalisées : {X_scaled.shape}")
print()

# ── Réduction de dimension pour visualisation (PCA 2D) ────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA : variance expliquée par les 2 composantes : {pca.explained_variance_ratio_.sum():.1%}")

# ── Réduction pour clustering (PCA haute dimension) ───────────────────────────
# Conserver 95% de la variance pour réduire le bruit avant clustering
pca_full = PCA(n_components=0.95, random_state=RANDOM_SEED)
X_reduced = pca_full.fit_transform(X_scaled)
print(f"PCA 95% variance : {X_reduced.shape[1]} composantes retenues")

---
## 🤖 Étape 5 — Comparatif de modèles de clustering

Nous testons **4 algorithmes** aux approches complémentaires :

| Algorithme | Type | Points forts en fintech |
|------------|------|------------------------|
| **K-Means** | Partitionnement | Rapide, interprétable, bon pour segments « ronds » |
| **HDBSCAN** | Densité | Détecte outliers, robuste aux formes complexes |
| **Agglomeratif** | Hiérarchique | Dendrogramme, utile pour analyses métier |
| **GMM** | Probabiliste | Probabilités d'appartenance, frontières « souples » |

### Choix du nombre de clusters (K-Means)
Nous utilisons la méthode **Elbow** (inertie) et le **score de silhouette** pour identifier le meilleur K.

In [ ]:
# ── Méthode Elbow + Silhouette pour K-Means ───────────────────────────────────
K_RANGE    = range(2, 9)
inertias   = []
sil_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = km.fit_predict(X_reduced)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_reduced, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(list(K_RANGE), inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Nombre de clusters (K)')
ax1.set_ylabel('Inertie')
ax1.set_title('Méthode Elbow — K-Means')
ax1.grid(alpha=0.3)

best_k = list(K_RANGE)[np.argmax(sil_scores)]
ax2.plot(list(K_RANGE), sil_scores, 'rs-', linewidth=2, markersize=8)
ax2.axvline(best_k, color='orange', linestyle='--', label=f'Meilleur K = {best_k}')
ax2.set_xlabel('Nombre de clusters (K)')
ax2.set_ylabel('Score de silhouette')
ax2.set_title('Score de silhouette — K-Means')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Sélection du nombre de clusters optimal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📌 K optimal selon le score de silhouette : K = {best_k}")
BEST_K = best_k

In [ ]:
# ── Entraînement des 4 modèles ────────────────────────────────────────────────
print(f"Entraînement des modèles (K = {BEST_K})...\n")

models = {}
labels_dict = {}

# 1. K-Means
km = KMeans(n_clusters=BEST_K, random_state=RANDOM_SEED, n_init=10)
labels_dict['K-Means'] = km.fit_predict(X_reduced)
models['K-Means'] = km
print(f"  ✅ K-Means         — {BEST_K} clusters")

# 2. HDBSCAN
hdb = hdbscan.HDBSCAN(
    min_cluster_size=max(10, len(X_reduced) // 50),
    min_samples=5,
    metric='euclidean',
    prediction_data=True,
)
labels_dict['HDBSCAN'] = hdb.fit_predict(X_reduced)
models['HDBSCAN'] = hdb
n_hdb = len(set(labels_dict['HDBSCAN'])) - (1 if -1 in labels_dict['HDBSCAN'] else 0)
n_noise = (labels_dict['HDBSCAN'] == -1).sum()
print(f"  ✅ HDBSCAN         — {n_hdb} clusters + {n_noise} points bruit")

# 3. Clustering agglomératif
agg = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
labels_dict['Agglomératif'] = agg.fit_predict(X_reduced)
models['Agglomératif'] = agg
print(f"  ✅ Agglomératif    — {BEST_K} clusters")

# 4. Gaussian Mixture Model
gmm = GaussianMixture(n_components=BEST_K, covariance_type='full', random_state=RANDOM_SEED, n_init=5)
labels_dict['GMM'] = gmm.fit_predict(X_reduced)
models['GMM'] = gmm
print(f"  ✅ GMM             — {BEST_K} composantes")

print("\n✅ Tous les modèles ont été entraînés.")

---
## 📏 Étape 6 — Évaluation des clusters

### Métriques utilisées

| Métrique | Objectif | Meilleure valeur |
|----------|----------|------------------|
| **Silhouette** | Cohésion intra-cluster vs séparation inter-cluster | → 1 (max) |
| **Davies-Bouldin** | Ratio similarité/séparation entre clusters | → 0 (min) |
| **Calinski-Harabasz** | Dispersion inter / intra clusters | → ∞ (max) |

> **Note** : Ces métriques supposent que tous les points sont assignés à un cluster.  
> Pour HDBSCAN, les points *bruit* (label = -1) sont **exclus** du calcul.

In [ ]:
def compute_metrics(X, labels):
    """Calcule les métriques de qualité en excluant les points bruit (label=-1)."""
    mask = labels != -1
    X_eval, lbl_eval = X[mask], labels[mask]

    if len(set(lbl_eval)) < 2:
        return {'silhouette': None, 'davies_bouldin': None, 'calinski_harabasz': None,
                'n_clusters': len(set(lbl_eval)), 'noise_ratio': (~mask).mean()}

    return {
        'silhouette':         round(silhouette_score(X_eval, lbl_eval), 4),
        'davies_bouldin':     round(davies_bouldin_score(X_eval, lbl_eval), 4),
        'calinski_harabasz':  round(calinski_harabasz_score(X_eval, lbl_eval), 2),
        'n_clusters':         len(set(lbl_eval)),
        'noise_ratio':        round((~mask).mean(), 4),
    }


results = {}
for model_name, labels in labels_dict.items():
    results[model_name] = compute_metrics(X_reduced, labels)

df_results = pd.DataFrame(results).T
print("\n📊 Récapitulatif des métriques d'évaluation :")
print("=" * 65)
display(df_results)

# Sélection du meilleur modèle (par silhouette)
best_sil = df_results['silhouette'].astype(float).idxmax()
print(f"\n🏆 Meilleur modèle selon le score de silhouette : {best_sil} ({df_results.loc[best_sil, 'silhouette']})")

In [ ]:
# ── Visualisation comparative des métriques ───────────────────────────────────
metric_cols = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
df_plot = df_results[metric_cols].astype(float)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']

for ax, col in zip(axes, metric_cols):
    bars = ax.bar(df_plot.index, df_plot[col], color=colors, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, df_plot[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Comparatif des métriques de clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📊 Étape 7 — Visualisations

### 7.1 Projection PCA 2D des clusters
Réduction en 2 dimensions pour visualiser la séparation des clusters.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for ax, (model_name, labels) in zip(axes, labels_dict.items()):
    unique_labels = sorted(set(labels))
    palette = sns.color_palette('tab10', max(len(unique_labels), 1))

    for i, lbl in enumerate(unique_labels):
        mask   = labels == lbl
        color  = 'lightgrey' if lbl == -1 else palette[i]
        marker = 'x' if lbl == -1 else 'o'
        label  = 'Bruit' if lbl == -1 else f'Cluster {lbl}'
        ax.scatter(
            X_pca[mask, 0], X_pca[mask, 1],
            c=[color], marker=marker, s=12,
            alpha=0.7, label=label,
        )

    sil = df_results.loc[model_name, 'silhouette']
    ax.set_title(f'{model_name}  (silhouette={sil})', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(loc='upper right', markerscale=2, fontsize=8)

plt.suptitle('Projection PCA 2D — Comparatif des algorithmes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Projection UMAP (⚙️ OPTIONNEL) ───────────────────────────────────────
# UMAP conserve mieux les structures locales que PCA.
# Peut être lent sur des grands datasets (>50k points).

if UMAP_AVAILABLE:
    print("⚙️  Calcul de la projection UMAP (peut prendre quelques secondes)...")
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=15, min_dist=0.1)
    X_umap  = reducer.fit_transform(X_scaled)

    # Affichage avec K-Means (meilleur modèle usuel)
    labels_km = labels_dict['K-Means']
    palette   = sns.color_palette('tab10', BEST_K)

    plt.figure(figsize=(9, 6))
    for lbl in range(BEST_K):
        mask = labels_km == lbl
        plt.scatter(X_umap[mask, 0], X_umap[mask, 1], c=[palette[lbl]],
                    s=12, alpha=0.7, label=f'Cluster {lbl}')
    plt.title('Projection UMAP — Clusters K-Means', fontsize=13, fontweight='bold')
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.legend(markerscale=3, fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️  UMAP non disponible — sautez cette cellule ou installez umap-learn.")

In [ ]:
# ── 7.3 Heatmap des profils clients par cluster (K-Means) ─────────────────────
# On analyse le cluster K-Means comme modèle de référence

SELECTED_MODEL = 'K-Means'  # ← changez ici pour analyser un autre modèle

df_features['cluster'] = labels_dict[SELECTED_MODEL]

# Profil moyen par cluster (features standardisées pour comparaison)
profile_cols  = FEATURE_COLS
cluster_means = df_features.groupby('cluster')[profile_cols].mean()

# Normalisation z-score pour l'affichage
cluster_means_std = (cluster_means - cluster_means.mean()) / (cluster_means.std() + 1e-9)

plt.figure(figsize=(13, 5))
sns.heatmap(
    cluster_means_std.T,
    cmap='RdBu_r', center=0,
    annot=True, fmt='.2f', linewidths=0.5,
    xticklabels=[f'Cluster {i}' for i in cluster_means.index],
)
plt.title(f'Profil moyen par cluster — {SELECTED_MODEL} (valeurs z-score)', fontsize=12, fontweight='bold')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 Distribution de la taille des clusters ────────────────────────────────
cluster_sizes = df_features['cluster'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Barplot
palette = sns.color_palette('tab10', len(cluster_sizes))
bars = ax1.bar([f'Cluster {i}' for i in cluster_sizes.index],
               cluster_sizes.values, color=palette)
ax1.set_title(f'Taille des clusters — {SELECTED_MODEL}', fontweight='bold')
ax1.set_ylabel('Nombre de clients')
for bar, val in zip(bars, cluster_sizes.values):
    pct = val / cluster_sizes.sum() * 100
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f'{val}\n({pct:.1f}%)', ha='center', fontsize=9)

# Pie chart
ax2.pie(cluster_sizes.values,
        labels=[f'Cluster {i}' for i in cluster_sizes.index],
        colors=palette, autopct='%1.1f%%', startangle=140)
ax2.set_title('Répartition en %', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.5 Boxplots des features clés par cluster ────────────────────────────────
KEY_FEATURES = ['recency_days', 'frequency', 'monetary', 'avg_amount',
                'international_ratio', 'amount_cv']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feat in zip(axes, KEY_FEATURES):
    data_by_cluster = [
        df_features.loc[df_features['cluster'] == c, feat].values
        for c in sorted(df_features['cluster'].unique())
    ]
    bp = ax.boxplot(data_by_cluster,
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xticklabels([f'C{i}' for i in sorted(df_features['cluster'].unique())])
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Distribution des features clés par cluster — {SELECTED_MODEL}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🏢 Étape 8 — Interprétation métier & Synthèse

### 8.1 Profils clients par cluster

Cette section agrège les statistiques clés de chaque cluster  
et propose une **interprétation business** basée sur les patterns observés.

In [ ]:
SUMMARY_COLS = [
    'recency_days', 'frequency', 'monetary',
    'avg_amount', 'debit_ratio', 'international_ratio',
    'active_days', 'amount_cv',
]

cluster_summary = df_features.groupby('cluster')[SUMMARY_COLS].agg(['mean', 'median']).round(2)
cluster_summary.columns = ['_'.join(c) for c in cluster_summary.columns]

print("\n📋 Profils statistiques des clusters :")
display(cluster_summary)

In [ ]:
# ── Labellisation automatique des clusters ────────────────────────────────────
# Règles heuristiques basées sur les statistiques observées.
# ⚠️  Adaptez ces seuils et labels à votre contexte métier !

def label_cluster(row):
    """
    Assigne un label métier à un cluster en fonction de son profil.
    Modifiez les seuils selon votre jeu de données.
    """
    rec  = row['recency_days_mean']
    freq = row['frequency_mean']
    mon  = row['monetary_mean']
    intl = row['international_ratio_mean']
    cv   = row['amount_cv_mean']

    if rec > 180 and freq < 5:
        return '😴 Clients inactifs'
    elif mon > cluster_summary['monetary_mean'].quantile(0.75) and freq > cluster_summary['frequency_mean'].median():
        return '⭐ Clients premium'
    elif intl > 0.3 or cv > 1.5:
        return '🚨 Profil atypique / Risque'
    else:
        return '👤 Clients standards'

cluster_summary['label_metier'] = cluster_summary.apply(label_cluster, axis=1)

print("\n🏷️  Labels métier attribués :")
for cluster_id, label in cluster_summary['label_metier'].items():
    size = cluster_sizes.get(cluster_id, 0)
    print(f"  Cluster {cluster_id} → {label}  ({size} clients)")

# Ajout du label dans le DataFrame principal
label_map = cluster_summary['label_metier'].to_dict()
df_features['cluster_label'] = df_features['cluster'].map(label_map)

In [ ]:
# ── 8.2 Radar chart des profils clients ───────────────────────────────────────
radar_features = ['recency_days', 'frequency', 'monetary', 'avg_amount',
                  'international_ratio', 'active_days']

# Normalisation 0-1 pour le radar
radar_data = df_features.groupby('cluster')[radar_features].mean()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

N      = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # fermer le polygone

fig, axes = plt.subplots(1, len(radar_norm), figsize=(4 * len(radar_norm), 4),
                         subplot_kw=dict(polar=True))
if len(radar_norm) == 1:
    axes = [axes]

for ax, (cluster_id, row) in zip(axes, radar_norm.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    color  = palette[int(cluster_id)]

    ax.fill(angles, values, color=color, alpha=0.25)
    ax.plot(angles, values, color=color, linewidth=2)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.replace('_', '\n') for f in radar_features], fontsize=8)
    ax.set_yticks([])
    lbl = label_map.get(cluster_id, f'Cluster {cluster_id}')
    ax.set_title(f'Cluster {cluster_id}\n{lbl}', fontweight='bold', fontsize=9, pad=12)

plt.suptitle('Radar — Profil moyen par cluster', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.3 Export des résultats ──────────────────────────────────────────────────
output_cols = ['customer_id', 'cluster', 'cluster_label'] + FEATURE_COLS
df_output   = df_features[output_cols].copy()

# Sauvegarde locale
OUTPUT_PATH = 'customer_clusters.csv'
df_output.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Résultats exportés → {OUTPUT_PATH}")
print(f"   {df_output.shape[0]} clients — {df_output['cluster'].nunique()} clusters")

# Aperçu
display(df_output.head(10))

---
## ✅ Synthèse & Recommandations

### Ce que nous avons fait

| Étape | Résultat |
|-------|----------|
| Génération / chargement | Dataset transactionnel fintech |
| Nettoyage | Suppression valeurs aberrantes, winsorisation, dédoublonnage |
| Feature engineering | RFM + 9 features comportementales + indicateurs anomalie |
| Prétraitement | RobustScaler + PCA (95% variance) |
| Clustering | K-Means, HDBSCAN, Agglomératif, GMM |
| Évaluation | Silhouette, Davies-Bouldin, Calinski-Harabasz |
| Visualisations | PCA 2D, UMAP, heatmap, boxplots, radar |
| Interprétation | Labels métier automatiques |

### Recommandations pour aller plus loin

1. **Données réelles** : remplacez le dataset synthétique par votre propre export CSV.  
2. **Features enrichies** : ajoutez des features contextuelles (ancienneté compte, score crédit, canal d'acquisition).  
3. **Stabilité temporelle** : re-exécutez le clustering sur différentes fenêtres temporelles et mesurez la stabilité des segments.  
4. **Monitoring** : mettez en place un score de drift pour détecter quand les segments évoluent.  
5. **Actions métier** :
   - *Clients inactifs* → campagne de réactivation
   - *Clients premium* → offres exclusives, programme de fidélité
   - *Profil atypique* → revue manuelle, analyse de risque
   - *Clients standards* → cross-sell produits courants

---
*Notebook créé dans le cadre du projet **D-fi-de-la-nuit-2024**.*  
*Compatible Google Colab et environnement local Python 3.8+.*